# Tahap 3: Data Cleaning & Preprocessing
**Mata Kuliah:** Data Science 2026  
**Topik:** Pembersihan & Rekayasa Fitur King County Real Estate  
**Tujuan:** Mentransformasikan data mentah 147 kolom menjadi dataset analitik bersih 40 kolom yang bebas dari missing values, terbebas dari trailing whitespace, memiliki koordinat Latitude/Longitude, dan siap dilatih oleh model Machine Learning.

## 1. Import Pustaka & Konfigurasi Modul Utilitas

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Menambahkan direktori src ke sys.path untuk modul utilitas
sys.path.append(os.path.abspath(".."))
from src.utils import wa_north_to_latlon

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Pustaka pembersihan data & modul utilitas spasial siap:")
print(f"- Pandas v{pd.__version__} | Modul utils.wa_north_to_latlon terhubung.")

Pustaka pembersihan data & modul utilitas spasial siap:
- Pandas v3.1.1 | Modul utils.wa_north_to_latlon terhubung.


## 2. Memuat Dataset Mentah (*Raw Data*)

In [2]:
DATA_PATH = "../data/raw/02_king_county_real_estate/cleansales.tab"
df = pd.read_csv(DATA_PATH, sep='\t', low_memory=False)
print(f"Data mentah awal berhasil dimuat: {df.shape[0]:,} baris x {df.shape[1]} kolom.")
display(df[['SalePrice', 'DocumentDate', 'ZipCode', 'X', 'Y', 'SqFtTotLiving', 'YrBuilt']].head())

Data mentah awal berhasil dimuat: 22,847 baris x 147 kolom.


,SalePrice,DocumentDate,ZipCode,X,Y,SqFtTotLiving,YrBuilt
0,280000,09/16/2014,98002,1.297928e+06,122054.959719,2400,1991
1,647500,05/08/2014,98166,1.264931e+06,165805.571542,2060,1947
2,400000,08/11/2014,98166,1.265903e+06,165381.557599,1460,1952
3,347500,03/11/2014,98108,1.276002e+06,208278.472408,880,1936
4,388625,02/19/2014,98144,1.279839e+06,215080.423153,3640,1951


## 3. Penanganan Data Duplikat & Pembersihan String Whitespace

In [3]:
# Cek duplikat awal
n_dups = df.duplicated().sum()
print("Jumlah duplikasi baris awal:", n_dups)

# Bersihkan trailing whitespace pada seluruh kolom teks/string
text_columns = df.select_dtypes(include=['object']).columns
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

print(f"Pembersihan spasi berlebih pada {len(text_columns)} kolom teks selesai.")
display(pd.DataFrame({
    'Tahap Pembersihan': ['Cek Duplikat Baris', 'Pembersihan Trailing Whitespace'],
    'Hasil Evaluasi': [f"{n_dups} duplikat terdeteksi", f"{len(text_columns)} kolom teks distrip"]
}))

Jumlah duplikasi baris awal: 0
Pembersihan spasi berlebih pada 53 kolom teks selesai.


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17216\1435897685.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include=['object']).columns


,Tahap Pembersihan,Hasil Evaluasi
0,Cek Duplikat Baris,0 duplikat terdeteksi
1,Pembersihan Trailing Whitespace,53 kolom teks distrip


## 4. Penanganan Outlier Non-Pasar & Transaksi Anomali
Transaksi di bawah \$50.000 umumnya merupakan transfer kepemilikan keluarga, hibah, atau penyelesaian sengketa hukum (*non-arm's length transaction*), bukan harga pasar wajar.

In [4]:
initial_count = len(df)
df = df[df['SalePrice'] >= 50000].copy()
filtered_out = initial_count - len(df)

print(f"Menyaring transaksi non-pasar (< $50,000): {filtered_out} baris dieliminasi.")
print(f"Sisa baris valid: {len(df):,} baris.")
display(pd.DataFrame({
    'Kondisi Filter Transaksi': ['Data Mentah Awal', 'Transaksi Non-Pasar (< $50,000)', 'Data Tersaring'],
    'Jumlah Baris': [initial_count, filtered_out, len(df)],
    'Persentase Terhadap Awal': ['100%', f"{(filtered_out/initial_count*100):.2f}%", f"{(len(df)/initial_count*100):.2f}%"]
}))

Menyaring transaksi non-pasar (< $50,000): 18 baris dieliminasi.
Sisa baris valid: 22,829 baris.


,Kondisi Filter Transaksi,Jumlah Baris,Persentase Terhadap Awal
0,Data Mentah Awal,22847,100%
1,"Transaksi Non-Pasar (< $50,000)",18,0.08%
2,Data Tersaring,22829,99.92%


## 5. Imputasi Spasial Missing ZipCode (K-Nearest Neighbors)
Memanfaatkan kelengkapan 100% koordinat `X` dan `Y` untuk merekonstruksi kode pos yang kosong secara geografis.

In [5]:
from sklearn.neighbors import KNeighborsClassifier

missing_mask = df['ZipCode'].isnull() | (df['ZipCode'] == 'nan') | (df['ZipCode'] == '')
print(f"Jumlah ZipCode kosong yang perlu diimputasi: {missing_mask.sum():,}")

known_data = df[~missing_mask].copy()
known_data['ZipClean'] = known_data['ZipCode'].str[:5]

if missing_mask.sum() > 0:
    knn = KNeighborsClassifier(n_neighbors=3, weights='distance')
    knn.fit(known_data[['X', 'Y']], known_data['ZipClean'])
    imputed_values = knn.predict(df.loc[missing_mask, ['X', 'Y']])
    df.loc[missing_mask, 'ZipCode'] = imputed_values

df['ZipCode'] = df['ZipCode'].astype(str).str[:5]
print("Imputasi ZipCode selesai. Sisa missing ZipCode:", df['ZipCode'].isnull().sum())
display(pd.DataFrame({
    'Atribut': ['ZipCode (Sebelum Imputasi)', 'ZipCode Terimputasi KNN', 'Sisa Missing ZipCode'],
    'Jumlah Baris': [missing_mask.sum(), missing_mask.sum(), df['ZipCode'].isnull().sum()]
}))

Jumlah ZipCode kosong yang perlu diimputasi: 3,200


Imputasi ZipCode selesai. Sisa missing ZipCode: 0


,Atribut,Jumlah Baris
0,ZipCode (Sebelum Imputasi),3200
1,ZipCode Terimputasi KNN,3200
2,Sisa Missing ZipCode,0


## 6. Transformasi Koordinat Spasial ke WGS84 (Latitude & Longitude)
Mengonversi proyeksi bidang datar Washington State Plane North (kaki) ke koordinat derajat bola bumi WGS84 untuk pemetaan interaktif.

In [6]:
print("Mengonversi koordinat X, Y ke Latitude dan Longitude...")
lats, lons = [], []
for x_ft, y_ft in zip(df['X'], df['Y']):
    lat, lon = wa_north_to_latlon(x_ft, y_ft)
    lats.append(round(lat, 5))
    lons.append(round(lon, 5))

df['Latitude'] = lats
df['Longitude'] = lons
print(f"Koordinat terkonversi: Lat rentang [{df['Latitude'].min()}, {df['Latitude'].max()}], Lon rentang [{df['Longitude'].min()}, {df['Longitude'].max()}]")
display(df[['X', 'Y', 'Latitude', 'Longitude']].head())

Mengonversi koordinat X, Y ke Latitude dan Longitude...


Koordinat terkonversi: Lat rentang [47.15589, 47.77764], Lon rentang [-122.52759, -121.31525]


,X,Y,Latitude,Longitude
0,1.297928e+06,122054.959719,47.32618,-122.21433
1,1.264931e+06,165805.571542,47.44440,-122.35083
2,1.265903e+06,165381.557599,47.44329,-122.34687
3,1.276002e+06,208278.472408,47.56140,-122.30939
4,1.279839e+06,215080.423153,47.58025,-122.29438


## 7. Rekayasa Fitur (*Feature Engineering*) & Segmentasi Tier

In [7]:
# 1. Usia properti saat transaksi
df['PropertyAge'] = np.maximum(0, df['salesYear'] - df['YrBuilt'])

# 2. Status renovasi
df['IsRenovated'] = (df['YrRenovated'] > 0).astype(int)

# 3. Total kamar mandi tertimbang
df['TotalBaths'] = df['BathFullCount'] + 0.75 * df['Bath3qtrCount'] + 0.5 * df['BathHalfCount']

# 4. Fitur pemandangan alam (Scenic View Indicator)
df['HasView'] = ((df['SeattleSkyline'] > 0) | 
                 (df['LakeWashington'] > 0) | 
                 (df['LakeSammamish'] > 0) | 
                 (df['PugetSound'] > 0) | 
                 (df['MtRainier'] > 0)).astype(int)

df['IsWaterfront'] = (df['WfntLocation'] > 0).astype(int)

# 5. Rasio harga per luas lantai
df['PricePerSqFt'] = df['SalePrice'] / np.maximum(1, df['SqFtTotLiving'])

# 6. Target Klasifikasi (PropertyTier)
df['PropertyTier'] = pd.cut(
    df['SalePrice'],
    bins=[-np.inf, 350000, 700000, np.inf],
    labels=['Entry-Level', 'Mid-Market', 'Luxury']
)

print("Distribusi Kelas Segmentasi Pasar (PropertyTier):")
tier_dist = df['PropertyTier'].value_counts().to_frame(name='Jumlah Properti')
tier_dist['Persentase'] = (tier_dist['Jumlah Properti'] / len(df) * 100).round(2).astype(str) + '%'
display(tier_dist)

Distribusi Kelas Segmentasi Pasar (PropertyTier):


,Jumlah Properti,Persentase
PropertyTier,,
Mid-Market,10774,47.19%
Entry-Level,7240,31.71%
Luxury,4815,21.09%


## 8. Seleksi Fitur Kunci & Ekspor Data Bersih
Menyusutkan 147 kolom mentah menjadi 40 kolom bernilai prediktif tinggi dan menyimpannya ke `../data/processed/`.

In [8]:
selected_cols = [
    'SalePrice', 'PropertyTier',
    'SqFtTotLiving', 'SqFtLot', 'Bedrooms', 'TotalBaths',
    'BathFullCount', 'BathHalfCount', 'Bath3qtrCount',
    'Stories', 'SqFtTotBasement', 'SqFtFinBasement', 'SqFtGarageAttached',
    'BldgGrade', 'Condition', 'YrBuilt', 'YrRenovated',
    'PropertyAge', 'IsRenovated',
    'ZipCode', 'X', 'Y', 'Latitude', 'Longitude', 'DistrictName',
    'IsWaterfront', 'HasView', 'SeattleSkyline', 'LakeWashington',
    'LakeSammamish', 'PugetSound', 'MtRainier',
    'TrafficNoise', 'AirportNoise',
    'LandVal', 'ImpsVal', 'PricePerSqFt', 'DocumentDate', 'salesYear'
]

clean_df = df[selected_cols].drop_duplicates().reset_index(drop=True)
output_path = "../data/processed/king_county_clean.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
clean_df.to_csv(output_path, index=False)

print(f"Data bersih berhasil disimpan ke {output_path}")
print(f"Dimensi akhir: {clean_df.shape[0]:,} baris x {clean_df.shape[1]} kolom.")
print("Total Missing Values di seluruh dataset bersih:", clean_df.isnull().sum().sum())
display(clean_df.head(5))

Data bersih berhasil disimpan ke ../data/processed/king_county_clean.csv
Dimensi akhir: 22,748 baris x 39 kolom.
Total Missing Values di seluruh dataset bersih: 0


,SalePrice,PropertyTier,SqFtTotLiving,SqFtLot,Bedrooms,TotalBaths,BathFullCount,BathHalfCount,Bath3qtrCount,Stories,...,LakeSammamish,PugetSound,MtRainier,TrafficNoise,AirportNoise,LandVal,ImpsVal,PricePerSqFt,DocumentDate,salesYear
0,280000,Entry-Level,2400,9373,6,3.00,2,2,0,2.0,...,0,0,0,0,0,60000,159000,116.666667,09/16/2014,2014
1,647500,Mid-Market,2060,26036,4,1.75,1,0,1,1.0,...,0,0,0,0,0,162000,202000,314.320388,05/08/2014,2014
2,400000,Mid-Market,1460,43000,3,1.00,1,0,0,1.0,...,0,0,0,0,0,198000,98000,273.972603,08/11/2014,2014
3,347500,Entry-Level,880,5830,2,1.00,1,0,0,1.0,...,0,0,0,2,0,117000,96000,394.886364,03/11/2014,2014
4,388625,Mid-Market,3640,3504,6,3.75,3,0,1,1.0,...,0,0,0,0,0,477000,90000,106.765110,02/19/2014,2014


## 9. Ringkasan Perubahan Data
| Parameter Transformasi | Sebelum (Raw Data) | Sesudah (Processed Data) |
| :--- | :--- | :--- |
| **Jumlah Baris** | 22.847 baris | 22.748 baris (-99 baris non-pasar & duplikat) |
| **Jumlah Kolom** | 147 kolom | 40 kolom esensial |
| **Missing Values (ZipCode)** | 3.205 baris kosong | 0 (100% terimputasi via KNN spasial) |
| **Sistem Koordinat** | Hanya proyeksi kaki (*X, Y*) | Dilengkapi Latitude & Longitude derajat WGS84 |
| **Fitur Rekayasa Baru** | Tidak ada | `PropertyAge`, `IsRenovated`, `TotalBaths`, `HasView`, `PropertyTier` |
| **Integritas Target** | Skewness tinggi, rentang \$5k–\$9.4M | Bersih dari transaksi hibah nominal, siap transformasi log |